# P03B — Audit dan eksplorasi dataset

Jalankan P03A terlebih dahulu. Notebook memeriksa skema, ID, missingness, duplikasi, rentang, ringkasan, korelasi Pearson/Spearman, serta membuat salinan rusak sementara untuk latihan diagnosis.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from scipy.stats import pearsonr,spearmanr
p=Path('praktikum/bab03/data_sintesis_beton.csv');df=pd.read_csv(p);out=p.parent
print('shape',df.shape,'unique IDs',df.id_spesimen.nunique(),'exact duplicates',int(df.duplicated().sum()),'missing cells',int(df.isna().sum().sum()))
print('ranges');print(df.drop(columns='id_spesimen').agg(['min','max']).T.to_string())
print('age counts');print(df.umur_hari.value_counts().sort_index().to_string())
print('descriptive');print(df.drop(columns='id_spesimen').describe().T.round(3).to_string())
# Check derived ratio from rounded water/cement; tiny differences are due to rounding.
df['ratio_recalculated']=df.air_kg_m3/df.semen_kg_m3;df['ratio_rounding_delta']=df.rasio_air_semen-df.ratio_recalculated
print('ratio delta max abs',df.ratio_rounding_delta.abs().max())
rows=[]
for x in ['semen_kg_m3','air_kg_m3','rasio_air_semen','fly_ash_kg_m3','superplasticizer_kg_m3','agregat_kasar_kg_m3','agregat_halus_kg_m3','umur_hari']:
 r,pv=pearsonr(df[x],df.kuat_tekan_MPa);rs,ps=spearmanr(df[x],df.kuat_tekan_MPa);rows.append([x,r,rs])
corr=pd.DataFrame(rows,columns=['feature','pearson_r','spearman_r']);corr.to_csv(out/'P03_correlations_synthetic.csv',index=False);print('target association (description only)');print(corr.round(3).to_string(index=False))
# Simulate defects only in a disposable copy to practice diagnostics; never overwrite the clean CSV.
a=df.copy();a.loc[0:19,'air_kg_m3']=np.nan;a=pd.concat([a,a.iloc[[50]]],ignore_index=True);a.loc[100,'kuat_tekan_MPa']=a.kuat_tekan_MPa.max()+50
print('practice-copy missing',int(a.isna().sum().sum()),'duplicate rows',int(a.duplicated().sum()),'extreme target',a.kuat_tekan_MPa.max());print('The copy is not saved over source data.')

shape (1200, 10) unique IDs 1200 exact duplicates 0 missing cells 0
ranges
                           min       max
semen_kg_m3             250.20   480.000
air_kg_m3                78.00   382.200
rasio_air_semen           0.30     0.819
fly_ash_kg_m3             0.00   120.000
superplasticizer_kg_m3    0.07    18.000
agregat_kasar_kg_m3     850.10  1149.800
agregat_halus_kg_m3     600.30   899.200
umur_hari                 3.00    90.000
kuat_tekan_MPa            5.00    53.100
age counts
umur_hari
3     150
7     228
14    178
28    328
56    133
90    183
descriptive
                         count     mean     std     min      25%      50%       75%       max
semen_kg_m3             1200.0  365.384  64.976  250.20  309.925  364.800   420.750   480.000
air_kg_m3               1200.0  204.436  65.486   78.00  155.275  196.050   248.925   382.200
rasio_air_semen         1200.0    0.560   0.148    0.30    0.434    0.561     0.684     0.819
fly_ash_kg_m3           1200.0   60.106  35.13